In [5]:
import os
from pathlib import Path

import requests
from dotenv import load_dotenv

cwd = Path.cwd()
root_candidates = (cwd, *cwd.parents, cwd / "project")
PROJECT_DIR = next((path for path in root_candidates if (path / ".env").is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Impossibile trovare .env risalendo dalla cartella corrente")

load_dotenv(PROJECT_DIR / ".env")
INPUT_DIR = PROJECT_DIR / "docs"/"KB"  # Modifica se i documenti sono in un'altra cartella
API_URL = os.getenv("RAG_API_URL", "http://127.0.0.1:8000").rstrip("/")
API_KEY = os.environ["API_KEY"]
SUPPORTED_EXTENSIONS = {".bmp", ".docx", ".heif", ".html", ".jpeg", ".jpg", ".pdf", ".png", ".pptx", ".tif", ".tiff", ".xlsx"}

if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"Cartella documenti non trovata: {INPUT_DIR}")

files = sorted(
    path for path in INPUT_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
)
if not files:
    raise ValueError(f"Nessun documento supportato trovato in {INPUT_DIR}")

print(f"Pronti {len(files)} documenti da inviare a {API_URL}/ingest")

Pronti 4 documenti da inviare a http://127.0.0.1:8000/ingest


In [6]:
for number, path in enumerate(files, start=1):
    document_key = path.relative_to(INPUT_DIR).as_posix()
    print(f"[{number}/{len(files)}] {path.name}")
    try:
        with path.open("rb") as document:
            response = requests.post(
                f"{API_URL}/ingest",
                headers={"Authorization": f"Bearer {API_KEY}"},
                data={"document_key": document_key},
                files={"file": (path.name, document)},
                timeout=300,
            )
        if response.ok:
            result = response.json()
            print(f"  OK: {result.get('indexed_count', '?')} chunk indicizzati")
        else:
            print(f"  ERRORE {response.status_code}: {response.text}")
    except requests.RequestException as exc:
        print(f"  ERRORE richiesta: {exc}")

[1/4] Agentic Ai in organisation.pdf
  OK: 49 chunk indicizzati
[2/4] ENISA Threat Landscape 2025 Booklet.pdf
  OK: 7 chunk indicizzati
[3/4] Generative AI for Official Statistics HLG-MOS Report.pdf
  OK: 265 chunk indicizzati
[4/4] Report-IA-intelligenza-artificiale-per-il-sistema-Italia.pdf
  OK: 65 chunk indicizzati
